In [ ]:
!pip install -q transformers datasets torch
!pip install -q soundfile librosa

In [ ]:
from datasets import load_dataset
from IPython.display import Audio, display

In [ ]:
# 加载 LibriSpeech 的一个小型验证集
print("正在加载数据集...")
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

# 提取第 3 个样本（索引为 2）
sample = dataset[2]

print("\n--- 目标文本 (Target Text) ---")
print(sample["text"])

print("\n--- 原始音频 (请点击播放) ---")
display(Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))

正在加载数据集...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]


--- 目标文本 (Target Text) ---
HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND

--- 原始音频 (请点击播放) ---


In [ ]:
from transformers import pipeline
pipe_ctc = pipeline(
    "automatic-speech-recognition",
    model="facebook/wav2vec2-base-100h")


# 音频输入是一个字典，其中包含 array 键（存储音频的原始波形数据）和 sampling_rate 键（存储音频的采样率）。
audio_input = {
    "array": sample["audio"]["array"],
    "sampling_rate": sample["audio"]["sampling_rate"]
}

print("正在进行语音识别...")
prediction_ctc = pipe_ctc(audio_input)

print("\n--- Wav2Vec2 (CTC) 识别结果 ---")
print(prediction_ctc["text"])

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


正在进行语音识别...

--- Wav2Vec2 (CTC) 识别结果 ---
HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAUS AND ROSE BEEF LOOMING BEFORE US SIMALYIS DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND


```
HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAUS AND ROSE BEEF LOOMING BEFORE US SIMALYIS DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
```
由于纯声学的处理矿建，对于某些词汇，CTC 在进行预测时会发生错误

In [ ]:
import torch
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"正在使用 {device} 加载 Whisper 模型...")

# 实例化 Whisper 管道
pipe_whisper = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=device
)

print("正在提取纯净的音频数组...")
# ❗️：直接提取纯 numpy 数组!!!!
audio_array = sample["audio"]["array"]

print("正在进行语音识别 (这可能需要几秒钟)...")
prediction_whisper = pipe_whisper(
    audio_array,
    max_new_tokens=256,
    chunk_length_s=30
    )

print("\n--- Whisper (Seq2Seq) 识别结果 ---")
print(prediction_whisper["text"])

正在使用 cpu 加载 Whisper 模型...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of tra

正在提取纯净的音频数组...
正在进行语音识别 (这可能需要几秒钟)...


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_pr


--- Whisper (Seq2Seq) 识别结果 ---
 He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similarly is drawn from eating and its results occur most readily to the mind.


预测结果正确，且带有标点及大小写。

In [ ]:
dataset = load_dataset(
    "facebook/multilingual_librispeech", "spanish", split="dev", streaming=True
)
sample = next(iter(dataset))

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
print(sample)
print(sample["transcript"])
Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])

{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7a4906b35e20>, 'original_path': 'http://www.archive.org/download/isaias_1603_librivox/isaias_28_reinavalera_64kb.mp3', 'begin_time': 193.17, 'end_time': 205.18, 'transcript': 'y á los hijos de los extranjeros que se allegaren á jehová para ministrarle y que amaren el nombre de jehová para ser sus siervos á todos los que guardaren el sábado de profanarlo y abrazaren mi pacto', 'audio_duration': 12.01000000000002, 'speaker_id': '10367', 'chapter_id': '10282', 'file': '10367_10282_000000.opus', 'id': '10367_10282_000000'}
y á los hijos de los extranjeros que se allegaren á jehová para ministrarle y que amaren el nombre de jehová para ser sus siervos á todos los que guardaren el sábado de profanarlo y abrazaren mi pacto


In [ ]:
import torch
from transformers import pipeline

# 实例化 pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small"
)

# 直接提取 numpy 数组
audio_array = sample["audio"]["array"]

# 修复幻觉问题：明确指定语言为 spanish，并执行翻译任务
result = pipe(
    audio_array,
    max_new_tokens=256,
    generate_kwargs={
        "task": "translate",
        "language": "spanish"
    },
    return_timestamps=True
    # 音频虽然只有12s，但是这样设置可以防止因处理非标准长度报错
)
print(result["text"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor o

 I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I wish I I'll see you in the next video, take care of your health, take care of your health, take care of your health

In [ ]:
import torch
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1,
)

audio_array = sample["audio"]["array"]

result = pipe(
    audio_array,
    generate_kwargs={
        "task": "translate",
        "language": "spanish",
        "repetition_penalty": 1.25,      # 抑制重复，必须 > 1.0
        "no_repeat_ngram_size": 3,     # 禁止3-gram重复（关键防幻觉）
        "temperature": 0.0,
        # 移除了 condition_on_previous_text 和 compression_ratio_threshold
    },
    return_timestamps=True,
    # 长音频分块处理（如果12秒短音频也建议设置，避免边界问题）
    chunk_length_s=30,
    stride_length_s=5,
)

print("转录结果：", result["text"])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


转录结果：  I wish you the best of health and happiness in your, I just wanted to tell you that I love you.


In [ ]:
print("--- 原始数据集文本 (Original Spanish) ---")
print(sample["transcript"])

print("\n--- Whisper 翻译结果 (English Translation) ---")
print(result["text"])

--- 原始数据集文本 (Original Spanish) ---
y á los hijos de los extranjeros que se allegaren á jehová para ministrarle y que amaren el nombre de jehová para ser sus siervos á todos los que guardaren el sábado de profanarlo y abrazaren mi pacto

--- Whisper 翻译结果 (English Translation) ---
 I wish you the best of health and happiness in your, I just wanted to tell you that I love you.


In [ ]:
# 第一步：强制转录（不翻译），看能否得到正确的西班牙语原文
result_es = pipe(
    audio_array,
    generate_kwargs={
        "task": "transcribe",  # 只转录
        "language": "spanish",
        "repetition_penalty": 1.2,
        "temperature": 0.0,
    },
)

spanish_text = result_es["text"]
print(f"转录结果: {spanish_text}")

# 第二步：如果转录正确，使用专门的翻译模型（如 M2M100 或 NLLB）
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

inputs = tokenizer(spanish_text, return_tensors="pt")
translated = model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"])
english_text = tokenizer.batch_decode(translated, skip_special_tokens=True)[0]
print(f"翻译结果: {english_text}")

Passing `generation_config` together with generation-related arguments=({'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


ValueError: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.